# LeMoN – train and predict

This notebook trains the LeMoN Bayesian neural network on your own dataset and then predicts parameters **with uncertainties**, writing the results to a CSV file. It runs unchanged on **Google Colab** and **locally**.

**What you need** (plain `.npy` files):

| file | content |
|---|---|
| `images.npy` | `(N,)` array of FITS paths **or** `(N, H, W)` array of images |
| `labels.npy` | `(N, P)` array of target values (e.g. `Ein_rad, e1, e2`) – needed for training only |
| `ids.npy` | *(optional)* `(N,)` identifiers, copied to the output CSV |

**Quick start:** leave `USE_EXAMPLE_DATA = True` below to run everything on a tiny *dummy* dataset (noise images, no lens simulation) – this only checks that the code runs. Then set it to `False` and point the paths to your data.

See the [README](README.md) for the full documentation.

## 1. Environment setup

In [ ]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/fab-gentile/LeMoN.git"   # used on Colab only

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                        # optional: keep data/results on Drive
    if not os.path.isdir("lemon_bnn") and not os.path.isdir("LeMoN"):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    if os.path.isdir("LeMoN"):
        os.chdir("LeMoN")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "astropy", "pandas"], check=True)

# Find the repository root (the folder that contains `lemon_bnn/`) and make it importable.
REPO_ROOT = os.getcwd()
while not os.path.isdir(os.path.join(REPO_ROOT, "lemon_bnn")):
    parent = os.path.dirname(REPO_ROOT)
    if parent == REPO_ROOT:
        raise RuntimeError("Could not find the lemon_bnn folder: run the notebook from inside the repository.")
    REPO_ROOT = parent
sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import lemon_bnn as lb

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend(), "| repo:", REPO_ROOT)

## 2. Configuration

Edit this cell only. Every setting is documented in `lemon_bnn/config.py`.

In [ ]:
# ---------------------------------------------------------------- what to run
USE_EXAMPLE_DATA = True    # True: tiny dummy dataset to check that everything runs
RUN_TRAINING     = True    # False: skip training and load weights already in OUTPUT_DIR
RUN_INFERENCE    = True

# ---------------------------------------------------------------- your data
# (ignored when USE_EXAMPLE_DATA = True)
IMAGES_NPY = "/path/to/images.npy"      # FITS paths or an (N, H, W) image array
LABELS_NPY = "/path/to/labels.npy"      # (N, P) targets
IDS_NPY    = None                       # optional (N,) identifiers
DATA_ROOT  = None                       # folder prepended to relative FITS paths (or None)

# Optional: predict on a *separate* set instead of the held-out test split
NEW_IMAGES_NPY = None                   # e.g. "/path/to/new_images.npy"
NEW_LABELS_NPY = None                   # optional, only used to add `*_true` columns
NEW_IDS_NPY    = None

OUTPUT_DIR = os.path.join(REPO_ROOT, "results")   # weights, config, logs and CSV go here

# ---------------------------------------------------------------- model / training
cfg = lb.Config(
    param_names=("Ein_rad", "e1", "e2"),   # one name per label column
    architecture="resnet34",               # resnet18 | resnet34 | resnet50
    dropout_rate=0.05,
    batch_size=64,
    epochs=200,
    learning_rate=1e-4,
    n_mc_samples=100,                      # stochastic forward passes per image
    val_fraction=0.1,                      # None = legacy behaviour: test set used for validation
    test_fraction=0.1,
)

# ---------------------------------------------------------------- example-data override
if USE_EXAMPLE_DATA:
    sys.path.insert(0, os.path.join(REPO_ROOT, "examples"))
    from make_example_data import make_example_data
    EX_DIR = make_example_data(os.path.join(REPO_ROOT, "examples", "data"), n=40, size=64)
    IMAGES_NPY, LABELS_NPY = os.path.join(EX_DIR, "images.npy"), os.path.join(EX_DIR, "labels.npy")
    IDS_NPY, DATA_ROOT = os.path.join(EX_DIR, "ids.npy"), EX_DIR
    OUTPUT_DIR = os.path.join(REPO_ROOT, "results_example")
    cfg.architecture, cfg.batch_size, cfg.epochs = "resnet18", 8, 2
    cfg.n_mc_samples, cfg.predict_batch_size = 5, 8

cfg.data_root = DATA_ROOT
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output folder:", OUTPUT_DIR)

## 3. Load the data and split it

In [ ]:
images, labels, ids = lb.load_dataset(IMAGES_NPY, LABELS_NPY, IDS_NPY)
cfg.input_shape = tuple(lb.infer_input_shape(images, cfg.fits_hdu, cfg.data_root))
print(f"{len(images)} objects | image shape {cfg.input_shape} | {labels.shape[1]} target(s): {cfg.param_names}")

split = lb.split_indices(len(images), cfg.val_fraction, cfg.test_fraction, cfg.seed)
tr, va, te = split["train"], split["val"], split["test"]
print(f"train: {len(tr)} | validation: {len(va)} | test: {len(te)}")

## 4. Train

Skipped if `RUN_TRAINING = False`. Writes `weights.weights.h5`, `config.json` and `training_log.csv` to `OUTPUT_DIR`. Early stopping restores the best weights.

In [ ]:
if RUN_TRAINING:
    model, history = lb.train_model(
        cfg, images[tr], labels[tr], images[va], labels[va], OUTPUT_DIR, verbose=2
    )
    print("Saved weights and config to", OUTPUT_DIR)
else:
    print("Training skipped; using the weights found in", OUTPUT_DIR)

## 5. Predict with uncertainties

Runs `cfg.n_mc_samples` Monte-Carlo-dropout passes per image and writes `predictions.csv` with, for every parameter, the prediction and the epistemic, aleatoric and total standard deviations.

In [ ]:
if RUN_INFERENCE:
    if NEW_IMAGES_NPY:                       # a separate dataset
        x_new, y_new, id_new = lb.load_dataset(NEW_IMAGES_NPY, NEW_LABELS_NPY, NEW_IDS_NPY)
    else:                                    # the held-out test split
        x_new, y_new, id_new = images[te], labels[te], ids[te]

    df = lb.run_inference(
        x_new, OUTPUT_DIR, ids=id_new, y_true=y_new,
        cfg=lb.Config.load(os.path.join(OUTPUT_DIR, "config.json")),
        csv_name="predictions.csv",
        samples_name=None,                   # e.g. "mc_samples.npy" to also keep the raw samples
    )
    print("Wrote", os.path.join(OUTPUT_DIR, "predictions.csv"))
    display(df.head())